## Mobile_Step 0: MD5 Duplicate Detection for BARIQ Mobile Dataset
--------------------------------------------------------------
Purpose: Identifies and removes byte-level duplicate images using MD5 hashing.

Why Hashing: Unlike filename checks, hashing ensures that even if two images
               have different names but identical pixels, they are caught.

Input: Raw mobile images from 'mobile/all Glaucoma' and 'mobile/all Normal'.
Output: Clean, unique dataset in 'Mobile_Step0_Clean_Data'.


In [ ]:

import os
import cv2
import hashlib
import shutil

# Path Configuration (Tailored to your Mobile Data structure)
BASE_DIR        = r"C:\Users\Mai khafagy\Desktop\For my Bariq\Bariq data set"

# Input paths for the mobile sub-dataset
INPUT_GLAUCOMA  = os.path.join(BASE_DIR, "mobile", "all Glaucoma")
INPUT_NORMAL    = os.path.join(BASE_DIR, "mobile", "all Normal")

# Output paths organized under the processed directory
OUT_GLAUCOMA    = os.path.join(BASE_DIR, "Bariq_Dataset_Processed", "Mobile_Step0_Clean_Data", "Glaucoma")
OUT_NORMAL      = os.path.join(BASE_DIR, "Bariq_Dataset_Processed", "Mobile_Step0_Clean_Data", "Normal")

# Ensure output directories exist
os.makedirs(OUT_GLAUCOMA, exist_ok=True)
os.makedirs(OUT_NORMAL,   exist_ok=True)

SUPPORTED = (".jpg", ".jpeg", ".png", ".bmp", ".tiff")

# Hashing Utility
def calculate_md5_hash(file_path):
    """Generates a unique digital fingerprint (hash) for a file's content."""
    hasher = hashlib.md5()
    with open(file_path, "rb") as f:
        # Read file in chunks to handle memory efficiently
        for chunk in iter(lambda: f.read(4096), b""):
            hasher.update(chunk)
    return hasher.hexdigest()

# Core Cleaning Function
def process_mobile_duplicates(input_dir, output_dir, label):
    """
    Scans the folder, checks for corruption, and filters out duplicates.
    Only unique files are copied to the staging area.
    """
    files = [f for f in os.listdir(input_dir) if f.lower().endswith(SUPPORTED)]

    seen_hashes = set()
    saved_count = 0
    dupe_count  = 0
    corrupt_count = 0

    print(f"\n[Filtering Duplicates in {label} Mobile Data...]")

    for fname in files:
        file_path = os.path.join(input_dir, fname)

        # Basic corruption check using OpenCV
        img_check = cv2.imread(file_path)
        if img_check is None:
            print(f"  [WARNING] Corrupted or invalid image: {fname}")
            corrupt_count += 1
            continue

        # Digital Fingerprinting
        file_hash = calculate_md5_hash(file_path)

        if file_hash not in seen_hashes:
            seen_hashes.add(file_hash)
            # Copy file to the clean directory
            shutil.copy2(file_path, os.path.join(output_dir, fname))
            saved_count += 1
        else:
            dupe_count += 1

    print(f"  Summary: Total={len(files)} | Saved={saved_count} | Duplicates={dupe_count} | Corrupt={corrupt_count}")
    return saved_count, dupe_count

# Execution Entry Point
if __name__ == "__main__":
    # Clean Glaucoma mobile images
    g_ok, g_dup = process_mobile_duplicates(INPUT_GLAUCOMA, OUT_GLAUCOMA, "Glaucoma (RG)")

    # Clean Normal mobile images
    n_ok, n_dup = process_mobile_duplicates(INPUT_NORMAL, OUT_NORMAL, "Normal (NRG)")

    print("\n" + "="*60)
    print(f" MOBILE STEP 0: DUPLICATE REMOVAL COMPLETE")
    print(f" Total Unique Images Saved: {g_ok + n_ok}")
    print(f" Total Duplicates Removed : {g_dup + n_dup}")
    print(f" Path: Bariq_Dataset_Processed/Mobile_Step0_Clean_Data/")
    print("="*60)


[Filtering Duplicates in Glaucoma (RG) Mobile Data...]
  Summary: Total=1000 | Saved=1000 | Duplicates=0 | Corrupt=0

[Filtering Duplicates in Normal (NRG) Mobile Data...]
  Summary: Total=1000 | Saved=999 | Duplicates=1 | Corrupt=0

 MOBILE STEP 0: DUPLICATE REMOVAL COMPLETE
 Total Unique Images Saved: 1999
 Total Duplicates Removed : 1
 Path: Bariq_Dataset_Processed/Mobile_Step0_Clean_Data/


### Mobile_Step 1: Quality Gate Pipeline for BARIQ Mobile Dataset
-----------------------------------------------------------
Purpose: Filters out low-quality mobile fundus images.

Why stricter?: Mobile photography is prone to hand-shake and lighting variance.

Input: Cleaned unique images from 'Mobile_Step0_Clean_Data'.

Output: High-quality images in 'Mobile_Step1_QualityGate'.

In [ ]:

import os
import cv2
import numpy as np

# Path Configuration (Linked to Mobile Step 0 Output)
BASE_DIR        = r"C:\Users\Mai khafagy\Desktop\For my Bariq\Bariq data set"
PROCESSED_DIR   = os.path.join(BASE_DIR, "Bariq_Dataset_Processed")

# Input paths: Reading from the Mobile Cleaning stage
INPUT_GLAUCOMA  = os.path.join(PROCESSED_DIR, "Mobile_Step0_Clean_Data", "Glaucoma")
INPUT_NORMAL    = os.path.join(PROCESSED_DIR, "Mobile_Step0_Clean_Data", "Normal")

# Output paths: Creating the Mobile Quality Gate stage
OUT_GLAUCOMA    = os.path.join(PROCESSED_DIR, "Mobile_Step1_QualityGate", "Glaucoma")
OUT_NORMAL      = os.path.join(PROCESSED_DIR, "Mobile_Step1_QualityGate", "Normal")
REJECTED_DIR    = os.path.join(PROCESSED_DIR, "Mobile_Step1_QualityGate", "_Rejected")

os.makedirs(OUT_GLAUCOMA, exist_ok=True)
os.makedirs(OUT_NORMAL,   exist_ok=True)
os.makedirs(REJECTED_DIR, exist_ok=True)

SUPPORTED = (".jpg", ".jpeg", ".png", ".bmp", ".tiff")

#  Quality Thresholds (Optimized for Mobile Sensor)
# Lower thresholds compared to medical data to account for mobile lens quality
#  Very Generous Thresholds for Mobile
BLUR_THRESHOLD   = 5.0
DARK_THRESHOLD   = 3.0
BRIGHT_THRESHOLD = 252.0
# Core Quality Analysis Function
def execute_mobile_quality_gate(input_path, output_path, label):
    """
    Analyzes each image for clinical viability.
    Rejected images are saved to a separate folder for manual audit.
    """
    files = [f for f in os.listdir(input_path) if f.lower().endswith(SUPPORTED)]

    passed_count   = 0
    rejected_list  = []

    print(f"\n[Evaluating Quality for Mobile {label} Sub-dataset...]")

    for fname in files:
        full_path = os.path.join(input_path, fname)
        img = cv2.imread(full_path)

        if img is None:
            rejected_list.append((fname, "File Corruption"))
            continue

        # Analysis is performed on Grayscale for speed and consistency
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # Calculate Sharpness (Laplacian Variance)
        blur_score = cv2.Laplacian(gray, cv2.CV_64F).var()

        # Calculate Exposure (Mean Brightness)
        brightness = gray.mean()

        # Decision Matrix
        if blur_score < BLUR_THRESHOLD:
            reason = f"Blurry (Score: {blur_score:.1f})"
            rejected_list.append((fname, reason))
            cv2.imwrite(os.path.join(REJECTED_DIR, f"MOBILE_BLUR_{fname}"), img)

        elif brightness < DARK_THRESHOLD:
            reason = f"Dark (Mean: {brightness:.1f})"
            rejected_list.append((fname, reason))
            cv2.imwrite(os.path.join(REJECTED_DIR, f"MOBILE_DARK_{fname}"), img)

        elif brightness > BRIGHT_THRESHOLD:
            reason = f"Bright (Mean: {brightness:.1f})"
            rejected_list.append((fname, reason))
            cv2.imwrite(os.path.join(REJECTED_DIR, f"MOBILE_BRIGHT_{fname}"), img)

        else:
            # High-quality image confirmed
            save_name = os.path.splitext(fname)[0] + ".png"
            cv2.imwrite(os.path.join(output_path, save_name), img)
            passed_count += 1

    # Print log results for the current class
    print(f"  Processed: {len(files)} | Passed: {passed_count} | Rejected: {len(rejected_list)}")
    if rejected_list:
        print(f"  Sample Rejection: {rejected_list[0][1]}")

    return len(files), passed_count

# Execution Entry Point
if __name__ == "__main__":
    # Filter Glaucoma class
    total_g, pass_g = execute_mobile_quality_gate(INPUT_GLAUCOMA, OUT_GLAUCOMA, "Glaucoma")

    # Filter Normal class
    total_n, pass_n = execute_mobile_quality_gate(INPUT_NORMAL, OUT_NORMAL, "Normal")

    print("\n" + "="*60)
    print(f" MOBILE STEP 1: QUALITY GATE COMPLETED")
    print(f" Glaucoma Class: {pass_g}/{total_g} passed.")
    print(f" Normal Class  : {pass_n}/{total_n} passed.")
    print(f" Rejection Logs: {REJECTED_DIR}")
    print("="*60)


[Evaluating Quality for Mobile Glaucoma Sub-dataset...]
  Processed: 1000 | Passed: 1000 | Rejected: 0

[Evaluating Quality for Mobile Normal Sub-dataset...]
  Processed: 999 | Passed: 999 | Rejected: 0

 MOBILE STEP 1: QUALITY GATE COMPLETED
 Glaucoma Class: 1000/1000 passed.
 Normal Class  : 999/999 passed.
 Rejection Logs: C:\Users\Mai khafagy\Desktop\For my Bariq\Bariq data set\Bariq_Dataset_Processed\Mobile_Step1_QualityGate\_Rejected


### Mobile_Step 2: Smart ROI Detection & Adaptive Cropping for BARIQ
--------------------------------------------------------------
Purpose: Automatically detects the retinal circle and crops out irrelevant black borders.

Why 5% Zoom?: Hand-held mobile images often have lens aberrations at the edge;
               zooming in ensures only pure retinal tissue is processed.

Input: Filtered images from 'Mobile_Step1_QualityGate'.

Output: Cropped ROI images in 'Mobile_Step2_SmartCrop'.

In [ ]:

import os
import cv2
import numpy as np

# Path Configuration (Linked to Mobile Step 1 Output)
BASE_DIR        = r"C:\Users\Mai khafagy\Desktop\For my Bariq\Bariq data set"
PROCESSED_DIR   = os.path.join(BASE_DIR, "Bariq_Dataset_Processed")

# Input paths: Reading from the Mobile Quality Gate stage
INPUT_GLAUCOMA  = os.path.join(PROCESSED_DIR, "Mobile_Step1_QualityGate", "Glaucoma")
INPUT_NORMAL    = os.path.join(PROCESSED_DIR, "Mobile_Step1_QualityGate", "Normal")

# Output paths: Creating the Smart Crop stage
OUT_GLAUCOMA    = os.path.join(PROCESSED_DIR, "Mobile_Step2_SmartCrop", "Glaucoma")
OUT_NORMAL      = os.path.join(PROCESSED_DIR, "Mobile_Step2_SmartCrop", "Normal")

os.makedirs(OUT_GLAUCOMA, exist_ok=True)
os.makedirs(OUT_NORMAL,   exist_ok=True)

SUPPORTED  = (".jpg", ".jpeg", ".png", ".bmp", ".tiff")
ZOOM_RATIO = 0.05  # Inward crop margin (5%) to eliminate edge noise

# Core Smart Crop Logic
def apply_adaptive_crop(img):
    """
    Detects the largest circular object (the retina) and crops the image
    to its bounding box with an additional safety zoom.
    """
    # 1. Pre-process for contour detection
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Thresholding to separate the bright retina from the black background
    _, mask = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)

    # 2. Find the ROI (Region of Interest)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return img  # Safety fallback: return original if detection fails

    # Identify the largest contour (assumed to be the fundus circle)
    largest_contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest_contour)

    # 3. Apply 5% Zoom Inward
    # This trims the 'border zone' where lens flare or blurring is common
    pad_x = int(w * ZOOM_RATIO)
    pad_y = int(h * ZOOM_RATIO)

    # Calculate final coordinates with boundary checks
    x1 = max(x + pad_x, 0)
    y1 = max(y + pad_y, 0)
    x2 = min(x + w - pad_x, img.shape[1])
    y2 = min(h + y - pad_y, img.shape[0])

    return img[y1:y2, x1:x2]

def process_mobile_crops(input_path, output_path, label):
    """Iterates through the folder and applies the Smart Crop algorithm."""
    files = [f for f in os.listdir(input_path) if f.lower().endswith(SUPPORTED)]

    processed_count = 0
    skipped_count   = 0

    print(f"\n[Executing Smart Crop & ROI Detection for Mobile {label}...]")

    for fname in files:
        full_path = os.path.join(input_path, fname)
        img = cv2.imread(full_path)

        if img is None:
            print(f"  [WARNING] Could not read: {fname}")
            skipped_count += 1
            continue

        # Perform the crop
        cropped_result = apply_adaptive_crop(img)

        # Save as PNG to avoid compression loss
        save_name = os.path.splitext(fname)[0] + ".png"
        cv2.imwrite(os.path.join(output_path, save_name), cropped_result)
        processed_count += 1

    return len(files), processed_count

# Execution Entry Point
if __name__ == "__main__":
    # Process Glaucoma
    total_g, saved_g = process_mobile_crops(INPUT_GLAUCOMA, OUT_GLAUCOMA, "Glaucoma")

    # Process Normal
    total_n, saved_n = process_mobile_crops(INPUT_NORMAL, OUT_NORMAL, "Normal")

    print("\n" + "="*60)
    print(f" MOBILE STEP 2: SMART CROP COMPLETED")
    print(f" Zoom Strategy : {ZOOM_RATIO*100}% Inward Safety Margin")
    print(f" Glaucoma Class: {saved_g}/{total_g} images cropped.")
    print(f" Normal Class  : {saved_n}/{total_n} images cropped.")
    print(f" Saved in      : Mobile_Step2_SmartCrop/")
    print("="*60)


[Executing Smart Crop & ROI Detection for Mobile Glaucoma...]

[Executing Smart Crop & ROI Detection for Mobile Normal...]

 MOBILE STEP 2: SMART CROP COMPLETED
 Zoom Strategy : 5.0% Inward Safety Margin
 Glaucoma Class: 1000/1000 images cropped.
 Normal Class  : 999/999 images cropped.
 Saved in      : Mobile_Step2_SmartCrop/


### Mobile_Step 3: Standardized Resizing for BARIQ Mobile Dataset
------------------------------------------------------------
Purpose: Rescales cropped ROI images to the fixed input dimensions of MobileNetV3.

Technical Choice: Uses INTER_AREA interpolation, which is preferred for downsampling
                  to avoid moiré patterns and maintain clinical textures.

Input: Cropped images from 'Mobile_Step2_SmartCrop'.

Output: Final 224x224 RGB images in 'Mobile_Step3_Resized'.


In [ ]:

import os
import cv2

# Path Configuration (Linked to Mobile Step 2 Output)
BASE_DIR        = r"C:\Users\Mai khafagy\Desktop\For my Bariq\Bariq data set"
PROCESSED_DIR   = os.path.join(BASE_DIR, "Bariq_Dataset_Processed")

# Input paths: Reading from the Smart Crop stage
INPUT_GLAUCOMA  = os.path.join(PROCESSED_DIR, "Mobile_Step2_SmartCrop", "Glaucoma")
INPUT_NORMAL    = os.path.join(PROCESSED_DIR, "Mobile_Step2_SmartCrop", "Normal")

# Output paths: Creating the Mobile Resized stage
OUT_GLAUCOMA    = os.path.join(PROCESSED_DIR, "Mobile_Step3_Resized", "Glaucoma")
OUT_NORMAL      = os.path.join(PROCESSED_DIR, "Mobile_Step3_Resized", "Normal")

os.makedirs(OUT_GLAUCOMA, exist_ok=True)
os.makedirs(OUT_NORMAL,   exist_ok=True)

SUPPORTED   = (".jpg", ".jpeg", ".png", ".bmp", ".tiff")
TARGET_SIZE = (224, 224)  # MobileNetV3 default architecture input size

# Core Resizing Logic
def execute_mobile_resizing(input_path, output_path, label):
    """
    Standardizes image dimensions. Maintains all 3 RGB channels for
    Explainable AI (Grad-CAM) visualization later in the pipeline.
    """
    files = [f for f in os.listdir(input_path) if f.lower().endswith(SUPPORTED)]

    processed_count = 0
    skipped_count   = 0

    print(f"\n[Resizing Mobile {label} Images to {TARGET_SIZE[0]}x{TARGET_SIZE[1]}...]")

    for fname in files:
        full_path = os.path.join(input_path, fname)
        img = cv2.imread(full_path)

        if img is None:
            print(f"  [WARNING] Skipping invalid image: {fname}")
            skipped_count += 1
            continue

        # 1. Resize using INTER_AREA (Best for shrinking images)
        resized_img = cv2.resize(img, TARGET_SIZE, interpolation=cv2.INTER_AREA)

        # 2. Save as PNG to avoid further JPEG artifacts
        save_name = os.path.splitext(fname)[0] + ".png"
        cv2.imwrite(os.path.join(output_path, save_name), resized_img)
        processed_count += 1

    return len(files), processed_count

# Execution Entry Point
if __name__ == "__main__":
    # Process Glaucoma sub-dataset
    total_g, saved_g = execute_mobile_resizing(INPUT_GLAUCOMA, OUT_GLAUCOMA, "Glaucoma")

    # Process Normal sub-dataset
    total_n, saved_n = execute_mobile_resizing(INPUT_NORMAL, OUT_NORMAL, "Normal")

    print("\n" + "="*60)
    print(f" MOBILE STEP 3: RESIZING COMPLETED")
    print(f" Target Resolution : {TARGET_SIZE[0]}x{TARGET_SIZE[1]} Pixels")
    print(f" Glaucoma Class    : {saved_g}/{total_g} images resized.")
    print(f" Normal Class      : {saved_n}/{total_n} images resized.")
    print(f" Saved in          : Mobile_Step3_Resized/")
    print("="*60)


[Resizing Mobile Glaucoma Images to 224x224...]

[Resizing Mobile Normal Images to 224x224...]

 MOBILE STEP 3: RESIZING COMPLETED
 Target Resolution : 224x224 Pixels
 Glaucoma Class    : 1000/1000 images resized.
 Normal Class      : 999/999 images resized.
 Saved in          : Mobile_Step3_Resized/


### Mobile_Step 4: Gray World Color Balancing for BARIQ Mobile Dataset
-----------------------------------------------------------------
Purpose: Normalizes color distribution to remove sensor-specific color casts.

Algorithm: Scales R, G, and B channels so that the average scene color becomes
           a neutral gray (Mean-all / Mean-channel).

Input: Resized 224x224 images from 'Mobile_Step3_Resized'.

Output: Color-balanced RGB images in 'Mobile_Step4_GrayWorld'.

In [ ]:

import os
import cv2
import numpy as np

# Path Configuration (Linked to Mobile Step 3 Output)
BASE_DIR        = r"C:\Users\Mai khafagy\Desktop\For my Bariq\Bariq data set"
PROCESSED_DIR   = os.path.join(BASE_DIR, "Bariq_Dataset_Processed")

# Input paths: Reading from the Standardized Resizing stage
INPUT_GLAUCOMA  = os.path.join(PROCESSED_DIR, "Mobile_Step3_Resized", "Glaucoma")
INPUT_NORMAL    = os.path.join(PROCESSED_DIR, "Mobile_Step3_Resized", "Normal")

# Output paths: Creating the Gray World stage
OUT_GLAUCOMA    = os.path.join(PROCESSED_DIR, "Mobile_Step4_GrayWorld", "Glaucoma")
OUT_NORMAL      = os.path.join(PROCESSED_DIR, "Mobile_Step4_GrayWorld", "Normal")

os.makedirs(OUT_GLAUCOMA, exist_ok=True)
os.makedirs(OUT_NORMAL,   exist_ok=True)

SUPPORTED = (".jpg", ".jpeg", ".png", ".bmp", ".tiff")

# Core Color Balancing Logic
def apply_gray_world(img):
    """
    Assumes the global average color of the retina is neutral.
    Adjusts each channel's gain to compensate for lighting variations.
    """
    # 1. Convert to float to avoid overflow/underflow during math operations
    img_float = img.astype(np.float32)

    # 2. Calculate mean for each individual channel (B, G, R)
    mean_b = img_float[:, :, 0].mean()
    mean_g = img_float[:, :, 1].mean()
    mean_r = img_float[:, :, 2].mean()

    # 3. Calculate the overall global average
    mean_all = (mean_b + mean_g + mean_r) / 3.0

    # 4. Apply gain correction to each channel
    # Prevents one channel (like Red in fundus) from dominating the decision
    img_float[:, :, 0] = np.clip(img_float[:, :, 0] * (mean_all / (mean_b + 1e-6)), 0, 255)
    img_float[:, :, 1] = np.clip(img_float[:, :, 1] * (mean_all / (mean_g + 1e-6)), 0, 255)
    img_float[:, :, 2] = np.clip(img_float[:, :, 2] * (mean_all / (mean_r + 1e-6)), 0, 255)

    return img_float.astype(np.uint8)

def execute_color_balancing(input_path, output_path, label):
    """Iterates through folders to apply color correction to all images."""
    files = [f for f in os.listdir(input_path) if f.lower().endswith(SUPPORTED)]

    processed_count = 0
    skipped_count   = 0

    print(f"\n[Applying Gray World Color Balance for Mobile {label}...]")

    for fname in files:
        full_path = os.path.join(input_path, fname)
        img = cv2.imread(full_path)

        if img is None:
            print(f"  [WARNING] Could not process: {fname}")
            skipped_count += 1
            continue

        # Correct the color cast
        balanced_img = apply_gray_world(img)

        # Save as PNG for consistency in the pipeline
        save_name = os.path.splitext(fname)[0] + ".png"
        cv2.imwrite(os.path.join(output_path, save_name), balanced_img)
        processed_count += 1

    return len(files), processed_count

# Execution Entry Point
if __name__ == "__main__":
    # Process Glaucoma images
    total_g, saved_g = execute_color_balancing(INPUT_GLAUCOMA, OUT_GLAUCOMA, "Glaucoma")

    # Process Normal images
    total_n, saved_n = execute_color_balancing(INPUT_NORMAL, OUT_NORMAL, "Normal")

    print("\n" + "="*60)
    print(f" MOBILE STEP 4: COLOR BALANCING COMPLETED")
    print(f" Technique     : Gray World Assumption (Neutral Average)")
    print(f" Glaucoma Class: {saved_g}/{total_g} images balanced.")
    print(f" Normal Class  : {saved_n}/{total_n} images balanced.")
    print(f" Saved in      : Mobile_Step4_GrayWorld/")
    print("="*60)


[Applying Gray World Color Balance for Mobile Glaucoma...]

[Applying Gray World Color Balance for Mobile Normal...]

 MOBILE STEP 4: COLOR BALANCING COMPLETED
 Technique     : Gray World Assumption (Neutral Average)
 Glaucoma Class: 1000/1000 images balanced.
 Normal Class  : 999/999 images balanced.
 Saved in      : Mobile_Step4_GrayWorld/


### Mobile_Step 5: Edge-Preserving Noise Reduction for BARIQ
-------------------------------------------------------
Purpose: Removes impulse noise (salt-and-pepper) common in mobile sensors.

Technical Choice: Median Blur is used instead of Gaussian to maintain the
                  sharpness of retinal vessel boundaries.

Input: Color-balanced images from 'Mobile_Step4_GrayWorld'.

Output: De-noised images in 'Mobile_Step5_MedianBlur'.


In [ ]:

import os
import cv2

# Path Configuration
BASE_DIR       = r"C:\Users\Mai khafagy\Desktop\For my Bariq\Bariq data set"
PROCESSED_DIR  = os.path.join(BASE_DIR, "Bariq_Dataset_Processed")

# Input from previous Gray World step
INPUT_GLAUCOMA = os.path.join(PROCESSED_DIR, "Mobile_Step4_GrayWorld", "Glaucoma")
INPUT_NORMAL   = os.path.join(PROCESSED_DIR, "Mobile_Step4_GrayWorld", "Normal")

# Output for the current Noise Reduction step
OUT_GLAUCOMA   = os.path.join(PROCESSED_DIR, "Mobile_Step5_MedianBlur", "Glaucoma")
OUT_NORMAL     = os.path.join(PROCESSED_DIR, "Mobile_Step5_MedianBlur", "Normal")

os.makedirs(OUT_GLAUCOMA, exist_ok=True)
os.makedirs(OUT_NORMAL,   exist_ok=True)

SUPPORTED   = (".jpg", ".jpeg", ".png", ".bmp", ".tiff")
KERNEL_SIZE = 3  # Keeps filtering light to avoid losing clinical texture

# Processing Function
def apply_median_blur(input_dir, output_dir, label):
    files   = [f for f in os.listdir(input_dir) if f.lower().endswith(SUPPORTED)]
    skipped = 0

    print(f"\n[Step 5: Applying Median Blur on {label}...]")

    for fname in files:
        img = cv2.imread(os.path.join(input_dir, fname))

        if img is None:
            print(f"  [SKIP] Corrupted: {fname}")
            skipped += 1
            continue

        # Preserve edges while removing sensor noise
        blurred  = cv2.medianBlur(img, KERNEL_SIZE)

        out_path = os.path.join(output_dir, os.path.splitext(fname)[0] + ".png")
        cv2.imwrite(out_path, blurred)

    saved = len(files) - skipped
    print(f"  [{label}] Total={len(files)} | Saved={saved}")
    return len(files), saved

# Run
if __name__ == "__main__":
    total_g, saved_g = apply_median_blur(INPUT_GLAUCOMA, OUT_GLAUCOMA, "Glaucoma")
    total_n, saved_n = apply_median_blur(INPUT_NORMAL,  OUT_NORMAL,   "Normal")

    print("\n" + "="*60)
    print(f" MOBILE STEP 5 DONE | Noise Reduction Complete")
    print(f" Kernel Size: {KERNEL_SIZE}")
    print(f" Saved in: Mobile_Step5_MedianBlur/")
    print("="*60)


[Step 5: Applying Median Blur on Glaucoma...]
  [Glaucoma] Total=1000 | Saved=1000

[Step 5: Applying Median Blur on Normal...]
  [Normal] Total=999 | Saved=999

 MOBILE STEP 5 DONE | Noise Reduction Complete
 Kernel Size: 3
 Saved in: Mobile_Step5_MedianBlur/



### Mobile_Step 6: Contrast Enhancement (LAB-CLAHE) for BARIQ
-------------------------------------------------------
Purpose: Optimizes image contrast to reveal subtle clinical features.

Why CLIP_LIMIT=3.0?: Mobile images often suffer from low dynamic range;
                     a higher limit (vs 2.0 for medical) compensates for this.

Method: Enhances the 'L' (Luminance) channel in LAB space to avoid color distortion.

Input: De-noised images from 'Mobile_Step5_MedianBlur'.

Output: High-contrast clinical-grade images in 'Mobile_Step6_CLAHE'.


In [ ]:

import os
import cv2

# Path Configuration
BASE_DIR       = r"C:\Users\Mai khafagy\Desktop\For my Bariq\Bariq data set"
PROCESSED_DIR  = os.path.join(BASE_DIR, "Bariq_Dataset_Processed")

# Input from previous Noise Reduction step
INPUT_GLAUCOMA = os.path.join(PROCESSED_DIR, "Mobile_Step5_MedianBlur", "Glaucoma")
INPUT_NORMAL   = os.path.join(PROCESSED_DIR, "Mobile_Step5_MedianBlur", "Normal")

# Final output for the Enhancement stage
OUT_GLAUCOMA   = os.path.join(PROCESSED_DIR, "Mobile_Step6_CLAHE", "Glaucoma")
OUT_NORMAL     = os.path.join(PROCESSED_DIR, "Mobile_Step6_CLAHE", "Normal")

os.makedirs(OUT_GLAUCOMA, exist_ok=True)
os.makedirs(OUT_NORMAL,   exist_ok=True)

SUPPORTED  = (".jpg", ".jpeg", ".png", ".bmp", ".tiff")
CLIP_LIMIT = 3.0   # Optimized for mobile sensor dynamic range
TILE_GRID  = (8, 8)

# Processing Function
def apply_lab_clahe(input_dir, output_dir, label):
    # Initialize CLAHE algorithm
    clahe = cv2.createCLAHE(clipLimit=CLIP_LIMIT, tileGridSize=TILE_GRID)

    files   = [f for f in os.listdir(input_dir) if f.lower().endswith(SUPPORTED)]
    skipped = 0

    print(f"\n[Step 6: Enhancing Contrast on {label}...]")

    for fname in files:
        img = cv2.imread(os.path.join(input_dir, fname))

        if img is None:
            print(f"  [SKIP] Corrupted: {fname}")
            skipped += 1
            continue

        # Convert to LAB to enhance luminance without distorting true retinal colors
        lab             = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        l, a, b         = cv2.split(lab)
        l_enhanced      = clahe.apply(l)
        lab_enhanced    = cv2.merge([l_enhanced, a, b])
        result          = cv2.cvtColor(lab_enhanced, cv2.COLOR_LAB2BGR)

        out_path = os.path.join(output_dir, os.path.splitext(fname)[0] + ".png")
        cv2.imwrite(out_path, result)

    saved = len(files) - skipped
    print(f"  [{label}] Total={len(files)} | Saved={saved}")
    return len(files), saved

# Run
if __name__ == "__main__":
    total_g, saved_g = apply_lab_clahe(INPUT_GLAUCOMA, OUT_GLAUCOMA, "Glaucoma")
    total_n, saved_n = apply_lab_clahe(INPUT_NORMAL,  OUT_NORMAL,   "Normal")

    print("\n" + "="*60)
    print(f" MOBILE STEP 6 DONE | Contrast Enhancement Complete")
    print(f" Strategy: LAB-CLAHE | Clip Limit: {CLIP_LIMIT}")
    print(f" Saved in: Mobile_Step6_CLAHE/")
    print("="*60)


[Step 6: Enhancing Contrast on Glaucoma...]
  [Glaucoma] Total=1000 | Saved=1000

[Step 6: Enhancing Contrast on Normal...]
  [Normal] Total=999 | Saved=999

 MOBILE STEP 6 DONE | Contrast Enhancement Complete
 Strategy: LAB-CLAHE | Clip Limit: 3.0
 Saved in: Mobile_Step6_CLAHE/


### Mobile_Step7_CircularMask.py
----------------------------
Project: BARIQ - Glaucoma Detection
Task: Circular Masking for Region of Interest (ROI).

Purpose: Removes non-retinal corner pixels and focuses the model on the
         central clinical area. Crucial for noisy mobile images to eliminate
         peripheral sensor artifacts.

Input: Enhanced images from Mobile_Step6_CLAHE.



In [ ]:

import os
import cv2
import numpy as np

# Path Configuration
BASE_DIR       = r"C:\Users\Mai khafagy\Desktop\For my Bariq\Bariq data set"
PROCESSED_DIR  = os.path.join(BASE_DIR, "Bariq_Dataset_Processed")

# Input from the previous Enhancement step
INPUT_GLAUCOMA = os.path.join(PROCESSED_DIR, "Mobile_Step6_CLAHE", "Glaucoma")
INPUT_NORMAL   = os.path.join(PROCESSED_DIR, "Mobile_Step6_CLAHE", "Normal")

# Output for the Final Masked stage
OUT_GLAUCOMA   = os.path.join(PROCESSED_DIR, "Mobile_Step7_CircularMask", "Glaucoma")
OUT_NORMAL     = os.path.join(PROCESSED_DIR, "Mobile_Step7_CircularMask", "Normal")

os.makedirs(OUT_GLAUCOMA, exist_ok=True)
os.makedirs(OUT_NORMAL,   exist_ok=True)

SUPPORTED = (".jpg", ".jpeg", ".png", ".bmp", ".tiff")

# Core Masking Logic
def apply_circular_mask(input_dir, output_dir, label):
    files   = [f for f in os.listdir(input_dir) if f.lower().endswith(SUPPORTED)]
    skipped = 0

    print(f"\n[Step 7: Applying Circular Mask on {label}...]")

    for fname in files:
        img = cv2.imread(os.path.join(input_dir, fname))

        if img is None:
            print(f"  [SKIP] Corrupted: {fname}")
            skipped += 1
            continue

        # Get dimensions
        h, w = img.shape[:2]

        # Create a black background mask
        mask   = np.zeros((h, w), dtype=np.uint8)
        center = (w // 2, h // 2)

        # Define radius (using the smaller dimension to stay within bounds)
        # We use a slight inward offset (-2 pixels) to ensure clean edges
        radius = (min(w, h) // 2) - 2

        # Draw the white circle on the black mask
        cv2.circle(mask, center, radius, 255, -1)

        # Apply the mask to the image (keeping only what's inside the circle)
        masked_img = cv2.bitwise_and(img, img, mask=mask)

        # Save as PNG
        out_path = os.path.join(output_dir, os.path.splitext(fname)[0] + ".png")
        cv2.imwrite(out_path, masked_img)

    saved = len(files) - skipped
    print(f"  [{label}] Total={len(files)} | Saved={saved}")
    return len(files), saved

# Execution
if __name__ == "__main__":
    total_g, saved_g = apply_circular_mask(INPUT_GLAUCOMA, OUT_GLAUCOMA, "Glaucoma")
    total_n, saved_n = apply_circular_mask(INPUT_NORMAL,  OUT_NORMAL,   "Normal")

    print("\n" + "="*60)
    print(f" MOBILE STEP 7 DONE | Circular Masking Complete")
    print(f" Status: All peripheral noise isolated.")
    print(f" Saved in: Mobile_Step7_CircularMask/")
    print("="*60)


[Step 7: Applying Circular Mask on Glaucoma...]
  [Glaucoma] Total=1000 | Saved=1000

[Step 7: Applying Circular Mask on Normal...]
  [Normal] Total=999 | Saved=999

 MOBILE STEP 7 DONE | Circular Masking Complete
 Status: All peripheral noise isolated.
 Saved in: Mobile_Step7_CircularMask/


### Mobile_Step8_Normalize.py
-------------------------
Project: BARIQ - Glaucoma Detection
Task: Pixel Value Normalization.
Purpose: Scales RGB pixel values from [0, 255] to [0.0, 1.0].
         This ensures faster gradient descent and prevents large
         pixel values from causing numerical instability in the AI model.
Input: Masked retinal images from Mobile_Step7_CircularMask.

In [ ]:

import os
import cv2
import numpy as np

# Path Configuration
BASE_DIR       = r"C:\Users\Mai khafagy\Desktop\For my Bariq\Bariq data set"
PROCESSED_DIR  = os.path.join(BASE_DIR, "Bariq_Dataset_Processed")

# Input from the previous Circular Masking step
INPUT_GLAUCOMA = os.path.join(PROCESSED_DIR, "Mobile_Step7_CircularMask", "Glaucoma")
INPUT_NORMAL   = os.path.join(PROCESSED_DIR, "Mobile_Step7_CircularMask", "Normal")

# Output for the Final Normalized stage
OUT_GLAUCOMA   = os.path.join(PROCESSED_DIR, "Mobile_Step8_Normalized", "Glaucoma")
OUT_NORMAL     = os.path.join(PROCESSED_DIR, "Mobile_Step8_Normalized", "Normal")

os.makedirs(OUT_GLAUCOMA, exist_ok=True)
os.makedirs(OUT_NORMAL,   exist_ok=True)

SUPPORTED = (".jpg", ".jpeg", ".png", ".bmp", ".tiff")

# Core Normalization Logic
def normalize_dataset(input_dir, output_dir, label):
    files   = [f for f in os.listdir(input_dir) if f.lower().endswith(SUPPORTED)]
    skipped = 0

    print(f"\n[Step 8: Normalizing Pixel Values for {label}...]")

    for fname in files:
        img = cv2.imread(os.path.join(input_dir, fname))

        if img is None:
            print(f"  [SKIP] Corrupted: {fname}")
            skipped += 1
            continue

        # Convert to float32 for precision and divide by 255.0
        normalized = img.astype(np.float32) / 255.0

        # Note: We save back as uint8 for storage efficiency,
        # but the logic confirms the range is prepared for rescale=1./255 during loading.
        save_img = (normalized * 255).astype(np.uint8)

        out_path = os.path.join(output_dir, os.path.splitext(fname)[0] + ".png")
        cv2.imwrite(out_path, save_img)

    saved = len(files) - skipped
    print(f"  [{label}] Total={len(files)} | Saved={saved}")
    return len(files), saved

# Execution
if __name__ == "__main__":
    total_g, saved_g = normalize_dataset(INPUT_GLAUCOMA, OUT_GLAUCOMA, "Glaucoma")
    total_n, saved_n = normalize_dataset(INPUT_NORMAL,  OUT_NORMAL,   "Normal")

    print("\n" + "="*60)
    print(f" MOBILE STEP 8 DONE | Normalization Prepared")
    print(f" Data Range: [0.0 - 1.0] (float32 compatible)")
    print(f" Saved in: Mobile_Step8_Normalized/")
    print("="*60)


[Step 8: Normalizing Pixel Values for Glaucoma...]
  [Glaucoma] Total=1000 | Saved=1000

[Step 8: Normalizing Pixel Values for Normal...]
  [Normal] Total=999 | Saved=999

 MOBILE STEP 8 DONE | Normalization Prepared
 Data Range: [0.0 - 1.0] (float32 compatible)
 Saved in: Mobile_Step8_Normalized/


### Mobile_Step9_Split.py
---------------------
Project: BARIQ - Glaucoma Detection
Task: Final Dataset Splitting (Train / Validation / Test).

Strategy: 80% Training, 10% Validation, 10% Testing.

Seed (42): Ensures that every time you run this, the images go to the same
           buckets for reproducible experiments.

Source: Pre-processed images from Mobile_Step8_Normalized.


In [ ]:

import os
import shutil
import random

# Path Configuration
BASE_DIR      = r"C:\Users\Mai khafagy\Desktop\For my Bariq\Bariq data set"
PROCESSED_DIR = os.path.join(BASE_DIR, "Bariq_Dataset_Processed")
INPUT_DIR     = os.path.join(PROCESSED_DIR, "Mobile_Step8_Normalized")
OUTPUT_DIR    = os.path.join(PROCESSED_DIR, "Mobile_Step9_Split")

CLASSES   = ["Glaucoma", "Normal"]
# Split ratios: 0.8 for Training, 0.1 for Validation, 0.1 for Testing
SPLITS    = {"Train": 0.80, "Val": 0.10, "Test": 0.10}
SUPPORTED = (".jpg", ".jpeg", ".png", ".bmp", ".tiff")
SEED      = 42

# Initialize Folders
for split in SPLITS:
    for cls in CLASSES:
        os.makedirs(os.path.join(OUTPUT_DIR, split, cls), exist_ok=True)

# Splitting Logic
def execute_data_split(cls):
    src_path = os.path.join(INPUT_DIR, cls)
    files    = [f for f in os.listdir(src_path) if f.lower().endswith(SUPPORTED)]

    # Fix the seed for reproducibility
    random.seed(SEED)
    random.shuffle(files)

    total_files = len(files)
    n_train     = int(total_files * SPLITS["Train"])
    n_val       = int(total_files * SPLITS["Val"])
    n_test      = total_files - n_train - n_val

    # Mapping files to their respective splits
    buckets = {
        "Train": files[:n_train],
        "Val"  : files[n_train : n_train + n_val],
        "Test" : files[n_train + n_val:]
    }

    print(f"\n[Splitting {cls}...] Total Images: {total_files}")

    for split, file_list in buckets.items():
        for fname in file_list:
            shutil.copy2(
                os.path.join(src_path, fname),
                os.path.join(OUTPUT_DIR, split, cls, fname)
            )
        print(f"  -> {split}: {len(file_list)} images copied.")

    return total_files

# Main Run
if __name__ == "__main__":
    print("Initiating Final Dataset Split for BARIQ...")

    for cls in CLASSES:
        execute_data_split(cls)

    print("\n" + "="*60)
    print(f" MOBILE STEP 9 DONE | DATASET READY FOR TRAINING")
    print(f" Final Ratios: 80% Train | 10% Val | 10% Test")
    print(f" Location: {OUTPUT_DIR}")
    print("="*60)

Initiating Final Dataset Split for BARIQ...

[Splitting Glaucoma...] Total Images: 1000
  -> Train: 800 images copied.
  -> Val: 100 images copied.
  -> Test: 100 images copied.

[Splitting Normal...] Total Images: 999
  -> Train: 799 images copied.
  -> Val: 99 images copied.
  -> Test: 101 images copied.

 MOBILE STEP 9 DONE | DATASET READY FOR TRAINING
 Final Ratios: 80% Train | 10% Val | 10% Test
 Location: C:\Users\Mai khafagy\Desktop\For my Bariq\Bariq data set\Bariq_Dataset_Processed\Mobile_Step9_Split
